
# EPLC Data Conversion Plan — Embedding Generator (multilingual-e5-base)

This Colab notebook will:
1. Install `sentence-transformers`  
2. Let you **upload** your `EPLC_Data_Conversion_Plan_Template.subsections_embedded.json` (skeleton created earlier)  
3. Generate embeddings with **intfloat/multilingual-e5-base** (L2-normalized)  
4. Let you **download** the final `*.embedded.json`

> Tip: If you don't have the skeleton yet, you can also upload the raw `EPLC_Data_Conversion_Plan_Template.json` and flip the switch in the next cell to auto-build the skeleton.


In [ ]:

#@title Setup — install and import
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
import json, pathlib, time, sys
from typing import List, Dict

# Model config
MODEL_NAME = "intfloat/multilingual-e5-base"
NORMALIZE = True
EMBED_DIM = 768  # expected
print("Using model:", MODEL_NAME)
model = SentenceTransformer(MODEL_NAME)
print("Model loaded.")


In [ ]:

#@title Upload input file(s)
# Choose ONE of the two ways:
# 1) Upload the SKELETON: EPLC_Data_Conversion_Plan_Template.subsections_embedded.json
# 2) Upload the RAW: EPLC_Data_Conversion_Plan_Template.json and set BUILD_SKELETON=True below

from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))


In [ ]:

#@title (Optional) Build skeleton from RAW JSON
BUILD_SKELETON = False  #@param {type:"boolean"}

import json, pathlib
from datetime import datetime

def build_skeleton_from_raw(raw_path: str, out_path: str):
    with open(raw_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    items = []
    for section in data.get("sections", []):
        if section.get("content"):
            items.append({
                "section_number": section.get("number"),
                "section_title": section.get("title"),
                "text": section.get("content"),
                "embedding": []
            })
        for sub in section.get("subsections", []) or []:
            if sub.get("content"):
                items.append({
                    "section_number": sub.get("number"),
                    "section_title": sub.get("title"),
                    "text": sub.get("content"),
                    "embedding": []
                })
    wrapped = {
        "document_title": data.get("document_title",""),
        "source_filename": pathlib.Path(raw_path).name,
        "embedding_model": MODEL_NAME,
        "embedding_dim": EMBED_DIM,
        "normalize_embeddings": True,
        "created_at": datetime.utcnow().isoformat()+"Z",
        "items": items
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(wrapped, f, ensure_ascii=False, indent=2)
    return out_path, len(items)

# Auto-detect uploaded files
names = [*uploaded.keys()]
skeleton_path = None
raw_path = None
for n in names:
    if n.endswith(".subsections_embedded.json"):
        skeleton_path = n
    if n.endswith(".json") and not n.endswith(".subsections_embedded.json"):
        raw_path = n

if BUILD_SKELETON:
    assert raw_path, "Please upload the RAW JSON first."
    skeleton_path, count = build_skeleton_from_raw(raw_path, "EPLC_Data_Conversion_Plan_Template.subsections_embedded.json")
    print(f"Built skeleton with {count} items ->", skeleton_path)
else:
    assert skeleton_path, "Please upload the skeleton JSON file (*.subsections_embedded.json) or enable BUILD_SKELETON."
    print("Skeleton detected:", skeleton_path)


In [ ]:

#@title Generate multilingual-e5-base embeddings
import json, numpy as np

with open(skeleton_path, "r", encoding="utf-8") as f:
    obj = json.load(f)

texts = [it.get("text","") for it in obj["items"]]

# Basic cleaning: remove placeholder-like text
def clean_text(t: str) -> str:
    t = t.strip()
    # remove simple placeholder markers like [Provide ...], [Insert ...]
    if t.startswith("[") and "]" in t and "Provide" in t or "Insert" in t:
        # keep as-is but it's still a valid string
        pass
    return t

texts = [clean_text(t) for t in texts]

embs = model.encode(texts, normalize_embeddings=NORMALIZE)

# sanity check
assert embs.shape[1] == EMBED_DIM, f"Unexpected embedding dimension: {embs.shape[1]} != {EMBED_DIM}"

for item, vec in zip(obj["items"], embs):
    item["embedding"] = vec.tolist()

print("Embedded items:", len(obj["items"]))
out_path = pathlib.Path(skeleton_path).with_suffix(".embedded.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(obj, f, ensure_ascii=False, indent=2)
print("Saved:", out_path)


In [ ]:

from google.colab import files
files.download(str(out_path))
print("Ready to download:", out_path)
